# Simuladores: Sistemas de Retroalimentación
### Capítulo 5 — *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Arturo Bouzas** · Facultad de Psicología, UNAM · bouzaslab25.com

---

Este notebook contiene tres simuladores del Capítulo 5:

| Simulador | Concepto central |
|-----------|-----------------|
| **1 — Fototaxia básica** | Comparación simultánea bilateral, ganancia, taxia positiva/negativa |
| **2 — Efecto de la demora** | Inestabilidad en sistemas de retroalimentación con latencia |
| **3 — Reactivo vs. Anticipatorio** | Límites de la retroalimentación pura; necesidad de predicción |

Corre las celdas en orden. Los simuladores interactivos requieren la celda de importaciones (Celda 1).


In [ ]:
# ── Celda 1: Importaciones ────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from collections import deque
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

# Activar gestor de widgets en Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # En Jupyter local no es necesario

# Paleta de colores usada en todos los simuladores
DARK_BG   = "#0d1117"
PANEL_BG  = "#161b22"
ACCENT1   = "#ff7832"   # naranja — explotación / error positivo
ACCENT2   = "#7ed6f8"   # azul    — exploración / error negativo
GOLD      = "#f0c040"   # dorado  — variable principal
TEXT      = "#c9d1d9"
SUBTEXT   = "#8b949e"
LINE      = "#30363d"

print("Importaciones completadas. Puedes ejecutar cualquiera de los tres simuladores.")


---
## Simulador 1 — Fototaxia Básica

Una polilla con dos sensores laterales de intensidad luminosa navega hacia (o lejos de) una fuente puntual.

**Ecuación de control:**
$$\omega = k \cdot (S_{der} - S_{izq})$$

donde $\omega$ es la tasa de giro, $k$ es la ganancia del sistema, y $S_{der} - S_{izq}$ es la señal de error.

**Ejercicios:**
- **Básico:** Con $\theta_0 = 75°$ y $k = 1.0$, observa la trayectoria. ¿Es una línea recta o una espiral? Si doblas $k$, ¿cambia el equilibrio o solo la velocidad?
- **Intermedio:** Activa "cubrir ojo izquierdo". ¿Qué ocurre? ¿Por qué el organismo gira continuamente?
- **Avanzado:** Cambia el signo de $k$ (fototaxia negativa). ¿Hacia dónde se mueve ahora el organismo?


In [ ]:
# ── Celda 2: Núcleo del Simulador 1 ──────────────────────────────────────

def simular_fototaxia(theta0_deg, k=1.0, dt=0.05, T=20.0,
                      ojo_cubierto=None, ruido=0.0):
    """
    Simula la orientación de un organismo con comparación simultánea bilateral.

    Parámetros
    ----------
    theta0_deg   : ángulo inicial respecto a la fuente (grados)
    k            : ganancia (k>0 = fototaxia positiva, k<0 = negativa)
    dt           : paso de tiempo
    T            : duración total de la simulación
    ojo_cubierto : None | 'izquierdo' | 'derecho'
    ruido        : desviación estándar del ruido en los sensores

    Retorna
    -------
    dict con arrays: theta (grados), omega, error, time
    """
    theta  = np.radians(theta0_deg)
    thetas = [np.degrees(theta)]
    omegas = [0.0]
    errors = [0.0]
    times  = [0.0]
    t      = 0.0

    while t < T:
        # Activación bilateral (proporcional a sin del ángulo en cada lado)
        # Sensor izquierdo detecta más cuando la fuente está a la izquierda
        s_izq = max(0.0, -np.sin(theta)) + np.random.normal(0, ruido)
        s_der = max(0.0,  np.sin(theta)) + np.random.normal(0, ruido)

        # Cubrir un ojo: el sensor cubierto devuelve cero
        if ojo_cubierto == 'izquierdo':
            s_izq = 0.0
        elif ojo_cubierto == 'derecho':
            s_der = 0.0

        # Señal de error y tasa de giro
        error = s_der - s_izq
        omega = k * error

        # Actualizar ángulo
        theta = theta - omega * dt
        # No restringir theta: permite que el organismo dé vueltas completas

        t += dt
        thetas.append(np.degrees(theta))
        omegas.append(omega)
        errors.append(error)
        times.append(t)

    return {
        "theta": np.array(thetas),
        "omega": np.array(omegas),
        "error": np.array(errors),
        "time":  np.array(times),
    }


def theta_to_xy(thetas_deg, speed=1.0, dt=0.05):
    """Convierte secuencia de ángulos en trayectoria (x, y)."""
    x, y = [0.0], [0.0]
    for theta_deg in thetas_deg[1:]:
        theta = np.radians(theta_deg)
        x.append(x[-1] + speed * np.cos(theta) * dt)
        y.append(y[-1] + speed * np.sin(theta) * dt)
    return np.array(x), np.array(y)


def plot_fototaxia(result, k, theta0_deg, ojo_cubierto, title=None):
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    for ax in axes:
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors=SUBTEXT)
        for sp in ax.spines.values(): sp.set_edgecolor(LINE)

    thetas = result["theta"]
    times  = result["time"]
    n      = len(thetas)
    xs, ys = theta_to_xy(thetas)

    # Posición de la fuente: siempre al este (1, 0) en el espacio del organismo
    fuente_x, fuente_y = max(xs) * 1.3 + 1.0, 0.0

    # ── Panel 1: Trayectoria ───────────────────────────────────────────────
    ax = axes[0]
    # Colorear la trayectoria por ángulo (naranja → frente a la fuente)
    for i in range(1, n):
        frac = 1.0 - abs(thetas[i]) / max(abs(thetas) + 1e-9)
        color = plt.cm.plasma(0.2 + 0.6 * frac)
        ax.plot([xs[i-1], xs[i]], [ys[i-1], ys[i]], color=color,
                lw=1.8, alpha=0.7 + 0.3 * (i / n))

    ax.scatter(xs[0], ys[0], s=90, c="white", zorder=6, label="Inicio")
    ax.scatter(xs[-1], ys[-1], s=160, c=ACCENT1, marker="*", zorder=6, label="Final")
    # Fuente de luz
    ax.scatter([fuente_x], [0], s=300, c=GOLD, marker="*", zorder=5,
               label="Fuente de luz")
    ax.axvline(fuente_x, color=GOLD, lw=0.6, ls=":", alpha=0.4)

    ax.set_title("Trayectoria del organismo", color=TEXT, fontsize=11)
    ax.set_xlabel("x", color=SUBTEXT); ax.set_ylabel("y", color=SUBTEXT)
    ax.legend(fontsize=8, framealpha=0.3, labelcolor="white",
              facecolor=DARK_BG, loc="upper left")

    # ── Panel 2: Ángulo θ en el tiempo ────────────────────────────────────
    ax = axes[1]
    ax.axhline(0, color=ACCENT1, lw=1.2, ls="--", alpha=0.7, label="Equilibrio (θ=0°)")
    ax.plot(times, thetas, color=TEXT, lw=1.3, alpha=0.9)
    ax.fill_between(times, thetas, 0,
                    where=(thetas > 0), color=ACCENT2, alpha=0.2)
    ax.fill_between(times, thetas, 0,
                    where=(thetas < 0), color=ACCENT1, alpha=0.2)
    ax.set_title("Ángulo θ(t) — grados", color=TEXT, fontsize=11)
    ax.set_xlabel("Tiempo", color=SUBTEXT)
    ax.set_ylabel("θ (grados)", color=SUBTEXT)
    ax.legend(fontsize=8, framealpha=0.3, labelcolor="white", facecolor=DARK_BG)
    ax.text(0.02, 0.97, f"ω = {k:.1f}·(S_der − S_izq)",
            transform=ax.transAxes, color=ACCENT2, fontsize=9, va="top",
            bbox=dict(boxstyle="round", facecolor=DARK_BG, alpha=0.7))

    # ── Panel 3: Señal de error ────────────────────────────────────────────
    ax = axes[2]
    ax.axhline(0, color=SUBTEXT, lw=0.8, ls="--", alpha=0.6)
    ax.plot(times, result["error"], color=GOLD, lw=1.2, alpha=0.9)
    ax.fill_between(times, result["error"], 0,
                    where=(result["error"] > 0), color=GOLD, alpha=0.25)
    ax.fill_between(times, result["error"], 0,
                    where=(result["error"] < 0), color=ACCENT2, alpha=0.25)
    ax.set_title("Señal de error (S_der − S_izq)", color=TEXT, fontsize=11)
    ax.set_xlabel("Tiempo", color=SUBTEXT)
    ax.set_ylabel("Error", color=SUBTEXT)

    ojo_str = f" | ojo {ojo_cubierto} cubierto" if ojo_cubierto else ""
    auto_title = title or (
        f"Fototaxia  |  k = {k},  θ₀ = {theta0_deg}°{ojo_str}"
    )
    fig.suptitle(auto_title, color=TEXT, fontsize=13, y=1.02)

    final_theta = thetas[-1] % 360
    equilibrio  = abs(thetas[-1]) < 5 or abs(abs(thetas[-1]) - 360) < 5
    fig.text(0.5, -0.03,
             f"θ final: {thetas[-1]:.1f}°  |  "
             f"{'Orientado ✓' if equilibrio else 'No orientado'}  |  "
             f"{'Sistema ABIERTO (ojo cubierto)' if ojo_cubierto else 'Sistema CERRADO'}",
             ha="center", color=SUBTEXT, fontsize=10)
    plt.tight_layout()
    plt.show()

# Prueba rápida
r = simular_fototaxia(theta0_deg=75, k=1.0)
plot_fototaxia(r, k=1.0, theta0_deg=75, ojo_cubierto=None)


In [ ]:
# ── Celda 3: Simulador 1 Interactivo ─────────────────────────────────────

style  = {"description_width": "210px"}
layout = widgets.Layout(width="500px")

sl_theta0 = widgets.FloatSlider(value=75, min=5, max=175, step=5,
                                description="θ₀ — Ángulo inicial (°)",
                                style=style, layout=layout)
sl_k      = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1,
                                description="k — Ganancia",
                                style=style, layout=layout,
                                readout_format=".1f")
sl_ruido  = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.02,
                                description="Ruido sensorial",
                                style=style, layout=layout,
                                readout_format=".2f")
dd_ojo    = widgets.Dropdown(
    options=["Ambos ojos (sistema cerrado)",
             "Cubrir ojo izquierdo (sistema abierto)",
             "Cubrir ojo derecho (sistema abierto)"],
    description="Condición experimental:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)

PRESETS_1 = {
    "— Experimentos del capítulo —":       None,
    "Fototaxia positiva, k moderado":      dict(theta0=75,  k=1.0,  ruido=0.0, ojo="Ambos ojos (sistema cerrado)"),
    "Fototaxia positiva, k alto":          dict(theta0=75,  k=3.0,  ruido=0.0, ojo="Ambos ojos (sistema cerrado)"),
    "Fototaxia positiva, k bajo":          dict(theta0=75,  k=0.3,  ruido=0.0, ojo="Ambos ojos (sistema cerrado)"),
    "Fototaxia negativa (huye de la luz)": dict(theta0=75,  k=-1.0, ruido=0.0, ojo="Ambos ojos (sistema cerrado)"),
    "Cubrir ojo izquierdo → giro continuo":dict(theta0=45,  k=1.0,  ruido=0.0, ojo="Cubrir ojo izquierdo (sistema abierto)"),
    "Cubrir ojo derecho → giro contrario": dict(theta0=45,  k=1.0,  ruido=0.0, ojo="Cubrir ojo derecho (sistema abierto)"),
    "Con ruido sensorial":                 dict(theta0=75,  k=1.0,  ruido=0.3, ojo="Ambos ojos (sistema cerrado)"),
}

dd_preset1 = widgets.Dropdown(options=list(PRESETS_1.keys()),
                              description="Experimento rápido:",
                              style={"description_width": "160px"},
                              layout=widgets.Layout(width="500px"))

btn_run1   = widgets.Button(description="▶  Ejecutar simulación",
                            button_style="success",
                            layout=widgets.Layout(width="230px"))
btn_reset1 = widgets.Button(description="↺  Parámetros por defecto",
                            button_style="warning",
                            layout=widgets.Layout(width="230px"))
out1 = widgets.Output()

def ojo_str_to_param(s):
    if "izquierdo" in s: return "izquierdo"
    if "derecho"   in s: return "derecho"
    return None

def run1(_=None):
    with out1:
        clear_output(wait=True)
        r = simular_fototaxia(theta0_deg=sl_theta0.value,
                              k=sl_k.value,
                              ojo_cubierto=ojo_str_to_param(dd_ojo.value),
                              ruido=sl_ruido.value)
        plot_fototaxia(r, k=sl_k.value, theta0_deg=sl_theta0.value,
                       ojo_cubierto=ojo_str_to_param(dd_ojo.value))

def reset1(_=None):
    sl_theta0.value = 75; sl_k.value = 1.0; sl_ruido.value = 0.0
    dd_ojo.value = "Ambos ojos (sistema cerrado)"
    dd_preset1.value = "— Experimentos del capítulo —"

def apply_preset1(change):
    p = PRESETS_1.get(change["new"])
    if p:
        sl_theta0.value = p["theta0"]; sl_k.value = p["k"]
        sl_ruido.value  = p["ruido"];  dd_ojo.value = p["ojo"]

btn_run1.on_click(run1)
btn_reset1.on_click(reset1)
dd_preset1.observe(apply_preset1, names="value")

display(widgets.VBox([sl_theta0, sl_k, sl_ruido, dd_ojo, dd_preset1,
                      widgets.HBox([btn_run1, btn_reset1])]), out1)
run1()


---
## Simulador 2 — Efecto de la Demora

Un sistema de retroalimentación puede volverse inestable si hay una demora entre el momento en que se detecta el error y el momento en que se ejecuta la corrección.

**Ejercicios:**
- **Básico:** Con $k = 1.0$ y sin demora, confirma la convergencia suave. Añade una demora de 10 pasos. ¿Qué ocurre?
- **Avanzado:** Encuentra el valor de demora a partir del cual el sistema comienza a oscilar para $k = 1.0$. Luego reduce $k$ a la mitad. ¿El mismo valor de demora ahora produce oscilaciones? ¿Por qué la ganancia y la demora interactúan?


In [ ]:
# ── Celda 4: Núcleo del Simulador 2 ──────────────────────────────────────

def simular_con_demora(theta0_deg, k=1.0, delay_steps=0, dt=0.05, T=25.0):
    """
    Simula el sistema de fototaxia con una demora entre detección y corrección.

    Parámetros
    ----------
    delay_steps : pasos de tiempo entre la detección del error y la corrección
    """
    theta  = np.radians(theta0_deg)
    buffer = deque([0.0] * max(delay_steps, 1), maxlen=max(delay_steps, 1))
    thetas = [np.degrees(theta)]
    times  = [0.0]
    t      = 0.0

    while t < T:
        error         = np.sin(theta)         # señal de error actual
        omega_actual  = buffer.popleft()       # corrección con demora
        buffer.append(k * error)              # nueva corrección encolada

        if delay_steps == 0:
            omega_actual = k * error           # sin demora: aplicar inmediatamente

        theta  = theta - omega_actual * dt
        t     += dt
        thetas.append(np.degrees(theta))
        times.append(t)

    return {"theta": np.array(thetas), "time": np.array(times)}


def plot_demora(configs, title=None):
    """Compara múltiples configuraciones de demora en una sola figura."""
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(DARK_BG)
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=SUBTEXT)
    for sp in ax.spines.values(): sp.set_edgecolor(LINE)

    colors = [ACCENT2, GOLD, ACCENT1, "#c084fc", "#34d399"]
    for i, cfg in enumerate(configs):
        r = simular_con_demora(**cfg)
        label = f"delay = {cfg.get('delay_steps', 0)} pasos  (k = {cfg.get('k', 1.0)})"
        ax.plot(r["time"], r["theta"], color=colors[i % len(colors)],
                lw=1.8, alpha=0.9, label=label)

    ax.axhline(0, color="white", lw=1.0, ls="--", alpha=0.5, label="Equilibrio (0°)")
    ax.set_xlabel("Tiempo", color=SUBTEXT)
    ax.set_ylabel("θ (grados)", color=SUBTEXT)
    ax.set_title(title or "Efecto de la demora en el sistema de retroalimentación",
                 color=TEXT, fontsize=12)
    ax.legend(fontsize=9, framealpha=0.3, labelcolor="white", facecolor=DARK_BG)
    plt.tight_layout()
    plt.show()

# Demostración inicial: sin demora vs. con demora
plot_demora([
    dict(theta0_deg=75, k=1.0, delay_steps=0),
    dict(theta0_deg=75, k=1.0, delay_steps=8),
    dict(theta0_deg=75, k=1.0, delay_steps=15),
], title="Sin demora vs. con demora (k = 1.0)")


In [ ]:
# ── Celda 5: Simulador 2 Interactivo ─────────────────────────────────────

style  = {"description_width": "210px"}
layout = widgets.Layout(width="500px")

sl_theta0b = widgets.FloatSlider(value=75, min=5, max=175, step=5,
                                 description="θ₀ — Ángulo inicial (°)",
                                 style=style, layout=layout)
sl_k2      = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                                 description="k — Ganancia",
                                 style=style, layout=layout,
                                 readout_format=".1f")
sl_delay   = widgets.IntSlider(  value=0, min=0, max=30, step=1,
                                 description="Demora (pasos)",
                                 style=style, layout=layout)

PRESETS_2 = {
    "— Experimentos del capítulo —":         None,
    "Sin demora (convergencia suave)":        dict(theta0=75, k=1.0,  delay=0),
    "Demora pequeña (leve oscilación)":       dict(theta0=75, k=1.0,  delay=8),
    "Demora grande (oscilaciones inestables)":dict(theta0=75, k=1.0,  delay=18),
    "Demora grande, k reducido":              dict(theta0=75, k=0.5,  delay=18),
    "k alto + demora → inestabilidad rápida": dict(theta0=75, k=2.5,  delay=6),
}

dd_preset2 = widgets.Dropdown(options=list(PRESETS_2.keys()),
                              description="Experimento rápido:",
                              style={"description_width": "160px"},
                              layout=widgets.Layout(width="500px"))

btn_run2   = widgets.Button(description="▶  Ejecutar simulación",
                            button_style="success",
                            layout=widgets.Layout(width="230px"))
btn_reset2 = widgets.Button(description="↺  Parámetros por defecto",
                            button_style="warning",
                            layout=widgets.Layout(width="230px"))
out2 = widgets.Output()

def run2(_=None):
    with out2:
        clear_output(wait=True)
        r = simular_con_demora(theta0_deg=sl_theta0b.value,
                               k=sl_k2.value,
                               delay_steps=sl_delay.value)
        fig, ax = plt.subplots(figsize=(10, 4.5))
        fig.patch.set_facecolor(DARK_BG)
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors=SUBTEXT)
        for sp in ax.spines.values(): sp.set_edgecolor(LINE)

        ax.plot(r["time"], r["theta"], color=GOLD, lw=1.5)
        ax.axhline(0, color="white", lw=1.0, ls="--", alpha=0.5,
                   label="Equilibrio (0°)")
        ax.fill_between(r["time"], r["theta"], 0,
                        where=(r["theta"] > 0), color=ACCENT2, alpha=0.2)
        ax.fill_between(r["time"], r["theta"], 0,
                        where=(r["theta"] < 0), color=ACCENT1, alpha=0.2)
        ax.set_xlabel("Tiempo", color=SUBTEXT)
        ax.set_ylabel("θ (grados)", color=SUBTEXT)
        ax.set_title(
            f"Efecto de la demora  |  k = {sl_k2.value},  "
            f"delay = {sl_delay.value} pasos,  θ₀ = {sl_theta0b.value}°",
            color=TEXT, fontsize=12)
        ax.legend(fontsize=9, framealpha=0.3, labelcolor="white", facecolor=DARK_BG)

        # Diagnostico automático
        last_half = r["theta"][len(r["theta"])//2:]
        oscila = np.std(last_half) > 5
        converge = abs(last_half[-1]) < 3
        status = "✓ Convergió" if converge else ("⚠ Oscila" if oscila else "○ No convergió")
        fig.text(0.5, -0.05,
                 f"Estado final: {status}  |  θ final = {r['theta'][-1]:.1f}°  |  "
                 f"Desviación estándar (segunda mitad) = {np.std(last_half):.1f}°",
                 ha="center", color=SUBTEXT, fontsize=10)
        plt.tight_layout()
        plt.show()

def reset2(_=None):
    sl_theta0b.value = 75; sl_k2.value = 1.0; sl_delay.value = 0
    dd_preset2.value = "— Experimentos del capítulo —"

def apply_preset2(change):
    p = PRESETS_2.get(change["new"])
    if p:
        sl_theta0b.value = p["theta0"]
        sl_k2.value      = p["k"]
        sl_delay.value   = p["delay"]

btn_run2.on_click(run2)
btn_reset2.on_click(reset2)
dd_preset2.observe(apply_preset2, names="value")

display(widgets.VBox([sl_theta0b, sl_k2, sl_delay, dd_preset2,
                      widgets.HBox([btn_run2, btn_reset2])]), out2)
run2()


---
## Simulador 3 — Reactivo vs. Anticipatorio

Dos conductores viajan en una ruta de 500 km con gasolineras distribuidas irregularmente.

- El **conductor reactivo** carga cuando el nivel del tanque cae por debajo de un umbral.
- El **conductor anticipatorio** aprende la distribución de gasolineras y carga *antes* de tramos largos.

**Ejercicios:**
- **Básico:** Con la distribución por defecto [40, 90, 250, 280, 450 km], ¿cuál conductor llega al final? ¿En qué kilómetro se queda varado el reactivo?
- **Avanzado:** Cambia las gasolineras a [20, 40, 60, 460, 480] —muchas al inicio, casi ninguna al final. ¿Puede el conductor reactivo tener éxito con algún umbral diferente? ¿Cuál es el umbral mínimo que garantiza llegar?


In [ ]:
# ── Celda 6: Núcleo del Simulador 3 ──────────────────────────────────────

def simular_conductor(gasolineras_km, autonomia_km=100,
                      reactivo=True, umbral_reactivo=0.15,
                      paso_km=5):
    """
    Simula un conductor en una ruta de 500 km.

    Parámetros
    ----------
    gasolineras_km  : lista con posiciones de gasolineras en km
    autonomia_km    : autonomía del tanque lleno
    reactivo        : True = estrategia reactiva, False = anticipatoria
    umbral_reactivo : fracción del tanque a partir de la cual carga (reactivo)
    paso_km         : resolución de la simulación en km
    """
    pos    = 0
    tanque = 1.0       # 1.0 = lleno
    consumo_por_km = 1.0 / autonomia_km

    historia_pos    = [0]
    historia_tanque = [tanque]
    historia_cargas = []
    varado = False

    while pos < 500:
        en_gasolinera = any(abs(pos - g) < paso_km / 2.0 for g in gasolineras_km)

        if reactivo:
            # Carga si está en gasolinera Y el tanque está bajo
            if en_gasolinera and tanque < umbral_reactivo:
                historia_cargas.append(pos)
                tanque = 1.0
        else:
            # Anticipatorio: carga si está en gasolinera Y necesitará más
            # gasolina antes de llegar a la siguiente que la disponible
            proximas = sorted(g for g in gasolineras_km if g > pos)
            km_prox  = (proximas[0] - pos) if proximas else (500 - pos)
            margen   = 0.10  # 10% extra de seguridad
            necesita = (km_prox * consumo_por_km) + margen
            if en_gasolinera and tanque < necesita:
                historia_cargas.append(pos)
                tanque = 1.0

        # Avanzar
        tanque -= consumo_por_km * paso_km
        pos    += paso_km

        if tanque <= 0:
            varado = True
            pos -= paso_km  # retrocede al último punto válido
            break

        historia_pos.append(pos)
        historia_tanque.append(max(tanque, 0.0))

    return {
        "pos":    np.array(historia_pos),
        "tanque": np.array(historia_tanque),
        "cargas": historia_cargas,
        "varado": varado,
        "km_final": pos,
    }


def plot_conductores(gasolineras_km, autonomia_km=100,
                     umbral_reactivo=0.15, paso_km=5):
    r_r = simular_conductor(gasolineras_km, autonomia_km,
                            reactivo=True,  umbral_reactivo=umbral_reactivo,
                            paso_km=paso_km)
    r_a = simular_conductor(gasolineras_km, autonomia_km,
                            reactivo=False, umbral_reactivo=umbral_reactivo,
                            paso_km=paso_km)

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    fig.patch.set_facecolor(DARK_BG)
    for ax in axes:
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors=SUBTEXT)
        for sp in ax.spines.values(): sp.set_edgecolor(LINE)

    for ax, r, label, color in zip(
        axes,
        [r_r, r_a],
        ["Conductor REACTIVO", "Conductor ANTICIPATORIO"],
        [ACCENT1, ACCENT2]
    ):
        ax.plot(r["pos"], r["tanque"] * 100, color=color, lw=2.0)
        ax.axhline(umbral_reactivo * 100, color="white", lw=0.8, ls=":",
                   alpha=0.5, label=f"Umbral reactivo ({umbral_reactivo*100:.0f}%)")
        ax.fill_between(r["pos"], r["tanque"] * 100, 0, color=color, alpha=0.2)

        # Marcar gasolineras
        for g in gasolineras_km:
            ax.axvline(g, color=GOLD, lw=0.8, ls="--", alpha=0.5)

        # Marcar cargas
        for km_carga in r["cargas"]:
            ax.axvline(km_carga, color="white", lw=1.5, alpha=0.8)
            ax.annotate("⛽", (km_carga, 95), ha="center", fontsize=9,
                        color="white")

        # Marcar si quedó varado
        if r["varado"]:
            ax.axvline(r["km_final"], color="red", lw=2.0)
            ax.text(r["km_final"] + 5, 50, "VARADO", color="red",
                    fontsize=10, fontweight="bold")

        status = "llegó (500 km)" if not r["varado"] else f"varado en km {r['km_final']:.0f}"
        ax.set_title(f"{label}  —  {status}", color=TEXT, fontsize=11)
        ax.set_ylabel("Nivel de tanque (%)", color=SUBTEXT)
        ax.set_ylim(0, 105)
        ax.legend(fontsize=9, framealpha=0.3, labelcolor="white",
                  facecolor=DARK_BG, loc="upper right")

    axes[-1].set_xlabel("Posición en la ruta (km)", color=SUBTEXT)

    gasolineras_str = ", ".join(str(int(g)) for g in gasolineras_km)
    fig.suptitle(f"Reactivo vs. Anticipatorio  |  Gasolineras en km: [{gasolineras_str}]",
                 color=TEXT, fontsize=12, y=1.02)

    cargas_r = len(r_r["cargas"]); cargas_a = len(r_a["cargas"])
    fig.text(0.5, -0.04,
             f"Cargas realizadas — reactivo: {cargas_r}  |  anticipatorio: {cargas_a}  "
             f"|  Gasolineras disponibles: {len(gasolineras_km)}  "
             f"|  Autonomía: {autonomia_km} km",
             ha="center", color=SUBTEXT, fontsize=10)
    plt.tight_layout()
    plt.show()

# Demostración inicial
plot_conductores(gasolineras_km=[40, 90, 250, 280, 450])


In [ ]:
# ── Celda 7: Simulador 3 Interactivo ─────────────────────────────────────

style  = {"description_width": "210px"}
layout = widgets.Layout(width="500px")

sl_autonomia = widgets.IntSlider(value=100, min=50, max=200, step=10,
                                 description="Autonomía del tanque (km)",
                                 style=style, layout=layout)
sl_umbral    = widgets.FloatSlider(value=0.15, min=0.05, max=0.60, step=0.05,
                                   description="Umbral reactivo (%)",
                                   style=style, layout=layout,
                                   readout_format=".0%")

# Input de texto para las gasolineras
txt_gasolineras = widgets.Text(
    value="40, 90, 250, 280, 450",
    description="Gasolineras (km, sep. por comas):",
    style={"description_width": "240px"},
    layout=widgets.Layout(width="520px"),
    placeholder="ej: 40, 90, 250, 280, 450"
)

PRESETS_3 = {
    "— Escenarios del capítulo —":               None,
    "Distribución por defecto (moderada)":        dict(g="40, 90, 250, 280, 450",  aut=100, umb=0.15),
    "Muchas al inicio, pocas al final":           dict(g="20, 40, 60, 460, 480",    aut=100, umb=0.15),
    "Distribuidas uniformemente":                 dict(g="100, 200, 300, 400",       aut=100, umb=0.15),
    "Un solo tramo muy largo en medio":           dict(g="50, 100, 400, 450",        aut=100, umb=0.15),
    "Autonomía reducida (60 km)":                 dict(g="40, 90, 250, 280, 450",   aut=60,  umb=0.15),
    "Umbral reactivo alto (40%)":                 dict(g="40, 90, 250, 280, 450",   aut=100, umb=0.40),
    "Sin gasolineras intermedias":                dict(g="0, 500",                   aut=100, umb=0.15),
}

dd_preset3 = widgets.Dropdown(options=list(PRESETS_3.keys()),
                              description="Escenario rápido:",
                              style={"description_width": "160px"},
                              layout=widgets.Layout(width="500px"))

btn_run3   = widgets.Button(description="▶  Ejecutar simulación",
                            button_style="success",
                            layout=widgets.Layout(width="230px"))
btn_reset3 = widgets.Button(description="↺  Parámetros por defecto",
                            button_style="warning",
                            layout=widgets.Layout(width="230px"))
out3 = widgets.Output()

def parse_gasolineras(txt):
    try:
        return sorted(float(x.strip()) for x in txt.split(",") if x.strip())
    except Exception:
        return [40, 90, 250, 280, 450]

def run3(_=None):
    with out3:
        clear_output(wait=True)
        gasolineras = parse_gasolineras(txt_gasolineras.value)
        plot_conductores(
            gasolineras_km=gasolineras,
            autonomia_km=sl_autonomia.value,
            umbral_reactivo=sl_umbral.value,
        )

def reset3(_=None):
    txt_gasolineras.value = "40, 90, 250, 280, 450"
    sl_autonomia.value = 100; sl_umbral.value = 0.15
    dd_preset3.value = "— Escenarios del capítulo —"

def apply_preset3(change):
    p = PRESETS_3.get(change["new"])
    if p:
        txt_gasolineras.value = p["g"]
        sl_autonomia.value    = p["aut"]
        sl_umbral.value       = p["umb"]

btn_run3.on_click(run3)
btn_reset3.on_click(reset3)
dd_preset3.observe(apply_preset3, names="value")

display(widgets.VBox([txt_gasolineras, sl_autonomia, sl_umbral, dd_preset3,
                      widgets.HBox([btn_run3, btn_reset3])]), out3)
run3()


## Ejercicio de cálculo manual (Ejercicio 3 del capítulo)

In [ ]:
# ── Celda 8: Ejercicio de cálculo manual ──────────────────────────────────
# Ejercicio 3 del capítulo:
# ω = k·(S_der − S_izq) con k = 2
# Verifica tu cálculo manual con esta celda.

k = 2.0

# Instante 1
s_der_1, s_izq_1 = 8.0, 5.0
error_1 = s_der_1 - s_izq_1
omega_1 = k * error_1
print(f"Instante 1: S_der={s_der_1}, S_izq={s_izq_1}")
print(f"  Error = {error_1:.1f}  →  ω = {omega_1:.1f}  →  La fuente está a la DERECHA")
print()

# Instante 2 (después del giro)
s_der_2, s_izq_2 = 6.5, 6.5
error_2 = s_der_2 - s_izq_2
omega_2 = k * error_2
print(f"Instante 2: S_der={s_der_2}, S_izq={s_izq_2}")
print(f"  Error = {error_2:.1f}  →  ω = {omega_2:.1f}  →  Equilibrio alcanzado")
print()
print("Reflexión:")
print("  En el instante 2 el error es exactamente 0.")
print("  Esto significa que el organismo está apuntando directamente hacia la fuente.")
print("  ¿Qué pasaría si hubiera un pequeño ruido en los sensores en ese momento?")


---
## Créditos y licencia

Este notebook es parte del proyecto:

> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*
> Lab25, Facultad de Psicología, UNAM.
> https://www.bouzaslab25.com

Código disponible en: **https://github.com/bouzaslab25/libro-aca**
Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
